# 04 — Validation: Can the Finance numbers be trusted?

Runs every Finance validation rule against **every layer** and stores the results, so each run leaves an audit trail that monitoring (`05_monitoring`) and alerting can read.

| Group | Rule | Why Finance cares |
|---|---|---|
| **V1 Order totals** | Every order total matches the sum of its line items (fixed tolerance **$0.01**); every order has at least one line item | `o_totalprice` is what was invoiced. If it does not equal its lines, either a line is missing or the header is wrong. |
| **V2 Discount & tax ranges** | `l_discount ∈ [0.00, 0.10]`, `l_tax ∈ [0.00, 0.08]`; the same bounds are declared as `CHECK` constraints in silver | An out-of-range rate silently changes revenue. Bounds were derived by profiling (`02_silver_profiling`, README). |
| **V3 Positive values** | `quantity`, `extended price`, `retail price` are strictly positive | A zero or negative price or quantity distorts revenue and AOV. |
| **V4 Cross-layer reconciliation** | The same net revenue (and gross revenue, order totals, row counts) computed in source → bronze → silver (+ quarantine) → gold | Proves no layer lost, duplicated or changed money. |

**Severity.** A rule on `silver` or `gold` is an **error**: those layers are what the business reads, so a failure fails the run (and the Job). The same rule on `bronze`/source is a **warning**: raw data is allowed to be dirty, because silver quarantines it, but we still want to see it. **Info** rules only report a number.

**Outputs** (schema `monitoring`, created if missing):

| Table | Mode | Content |
|---|---|---|
| `dq_results` | append | One row per rule × layer × run |
| `layer_reconciliation` | append | Money and row counts per layer, per run |
| `mismatched_orders_latest` | overwrite | Every order that currently fails V1, with the reason |
| `mismatched_orders_log` | append | The same, kept for every run (used to detect *new* mismatches) |
| `metric_mismatched_orders` | append | **Monitoring metric**: mismatched orders per run, how many are new, alert flag |

Run after `03_gold`. Widgets let a Job (or `06_alert_demo`) run only some rule groups and decide whether a failure should stop the run.

In [ ]:
# Parameters (override from a Job or the widget bar to run in another workspace)
dbutils.widgets.text("catalog", "group1_finance", "Catalog")
dbutils.widgets.text("source_catalog", "samples", "Source catalog")
dbutils.widgets.text("source_schema", "tpch", "Source schema")
dbutils.widgets.text("bronze_schema", "bronze", "Bronze schema")
dbutils.widgets.text("silver_schema", "silver", "Silver schema")
dbutils.widgets.text("gold_schema", "gold", "Gold schema")
dbutils.widgets.text("monitoring_schema", "monitoring", "Monitoring schema")
dbutils.widgets.text("run_label", "manual", "Run label (scheduled / manual / demo)")
dbutils.widgets.text("rule_groups", "all", "Rule groups (all, or comma list)")
dbutils.widgets.text("fail_on_error", "true", "Fail the run on an error rule")


def widget(name):
    return dbutils.widgets.get(name).strip()


CATALOG = widget("catalog")
SOURCE  = f"{widget('source_catalog')}.{widget('source_schema')}"
BRONZE  = f"{CATALOG}.{widget('bronze_schema')}"
SILVER  = f"{CATALOG}.{widget('silver_schema')}"
GOLD    = f"{CATALOG}.{widget('gold_schema')}"
MON     = f"{CATALOG}.{widget('monitoring_schema')}"

ALL_GROUPS = ["order_total", "ranges", "positive", "reconciliation"]
requested = {g.strip() for g in widget("rule_groups").split(",") if g.strip()}
unknown = requested - set(ALL_GROUPS) - {"all"}
assert not unknown, f"Unknown rule groups {sorted(unknown)}; valid: {ALL_GROUPS} or 'all'"
GROUPS = ALL_GROUPS if (not requested or "all" in requested) else [g for g in ALL_GROUPS if g in requested]

RUN_LABEL     = widget("run_label") or "manual"
FAIL_ON_ERROR = widget("fail_on_error").lower() in ("true", "1", "yes")

import json
import re
import uuid
from decimal import Decimal

from pyspark.sql import functions as F

RUN_ID = str(uuid.uuid4())
RUN_TS = spark.sql("SELECT current_timestamp() AS ts").first()["ts"]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {MON} COMMENT 'Finance validation results, monitoring metrics and alerts'")

print(f"Source     : {SOURCE}")
print(f"Layers     : {BRONZE} -> {SILVER} -> {GOLD}")
print(f"Results to : {MON}")
print(f"Run        : {RUN_ID}  label={RUN_LABEL}  at {RUN_TS}")
print(f"Groups     : {', '.join(GROUPS)}   fail_on_error={FAIL_ON_ERROR}")

## Business rules

One place for every threshold. The discount and tax bounds come from `02_silver_profiling` and **must stay equal** to the constants in `02_silver`; rule V2.3 checks that the `CHECK` constraints actually registered on `silver.lineitem` use these same bounds, so a drift between the two notebooks is caught automatically.

In [ ]:
ORDER_TOTAL_TOLERANCE = Decimal("0.01")   # fixed by the Finance requirement
MONEY_TOLERANCE       = Decimal("0.01")   # cross-layer reconciliation of money totals

DISCOUNT_MIN, DISCOUNT_MAX = Decimal("0.00"), Decimal("0.10")   # derived in 02_silver_profiling
TAX_MIN, TAX_MAX           = Decimal("0.00"), Decimal("0.08")   # derived in 02_silver_profiling

# Raw layers may be dirty (silver quarantines bad rows); curated layers must be clean.
LAYER_SEVERITY = {"source": "warn", "bronze": "warn", "silver": "error", "gold": "error"}
LAYER_ORDER    = {"source": 0, "bronze": 1, "silver": 2, "gold": 3, "all": 4}

In [ ]:
RESULTS = []
METRIC  = None   # filled by the order-total group


def table_exists(name):
    return spark.catalog.tableExists(name)


def money(x):
    return "n/a" if x is None else f"{Decimal(x):,.2f}"


def record(rule_id, group, layer, rule, severity, checked, failed, observed="", expected=""):
    checked, failed = int(checked or 0), int(failed or 0)
    if severity == "info":
        status = "INFO"
    elif failed == 0:
        status = "PASS"
    else:
        status = "FAIL" if severity == "error" else "WARN"
    RESULTS.append({
        "run_id": RUN_ID, "run_ts": RUN_TS, "run_label": RUN_LABEL,
        "rule_id": rule_id, "rule_group": group, "layer": layer, "rule": rule,
        "severity": severity, "status": status,
        "checked_rows": checked, "failed_rows": failed,
        "observed": observed, "expected": expected,
    })
    print(f"[{status:<4}] {rule_id:<5} {layer:<7} {rule}  (failed {failed:,} of {checked:,})" + (f"  | {observed}" if observed else ""))


def record_missing(rule_id, group, layer, rule, table):
    """A table the rule needs does not exist: that is itself a failure of the pipeline."""
    record(rule_id, group, layer, rule, LAYER_SEVERITY.get(layer, "error"), 0, 1, f"table not found: {table}")

## V1 — Order totals reconcile with their line items

**How `o_totalprice` is built.** Profiling showed that the obvious formula `Σ extendedprice × (1 − discount) × (1 + tax)` does **not** reproduce `o_totalprice`: most orders differ by a few cents (up to ~$0.11), always in the same direction. The reason is in the TPC-H generator (`dbgen`, `build.c`, `mk_order`): it computes the order total in **integer cents** and **truncates each line twice**, first after the discount and again after the tax:

```
line_cents = ((extprice_cents × (100 − discount%)) DIV 100) × (100 + tax%) DIV 100
o_totalprice = Σ line_cents / 100
```

That is ordinary invoicing: every line is billed in whole cents. We reproduce exactly that billing rule and keep the **fixed $0.01 tolerance** required by Finance. We do *not* widen the tolerance to absorb rounding: with the right rule the expected difference is exactly $0.00, so any order that is off by more than one cent is a real defect.

The naive formula is still reported (V1.3, info) so the size of the rounding effect is visible.

Orders are **left-joined** to their lines, so an order with **no line items** is not silently dropped: its line sum is $0 and it is reported both as a mismatch (V1.1) and on its own (V1.2).

In [ ]:
def order_reconciliation(schema):
    """One row per order in `schema`: recorded total vs. the sum of its line items."""
    return spark.sql(f"""
        WITH lines AS (
            SELECT
                l_orderkey,
                COUNT(*) AS line_count,
                -- billed amount per line, in whole cents (same integer arithmetic as dbgen)
                SUM(
                    ((CAST(l_extendedprice * 100 AS BIGINT) * (100 - CAST(l_discount * 100 AS INT))) DIV 100)
                    * (100 + CAST(l_tax * 100 AS INT)) DIV 100
                ) AS billed_cents,
                -- exact decimal arithmetic, no rounding (informational)
                SUM(l_extendedprice * (1 - l_discount) * (1 + l_tax)) AS naive_total
            FROM {schema}.lineitem
            GROUP BY l_orderkey
        ),
        joined AS (
            SELECT
                o.o_orderkey   AS order_key,
                o.o_orderdate  AS order_date,
                o.o_totalprice AS recorded_total,
                COALESCE(l.line_count, 0) AS line_count,
                CAST(COALESCE(l.billed_cents, 0) / 100.0 AS DECIMAL(18,2)) AS billed_total,
                CAST(COALESCE(l.naive_total, 0) AS DECIMAL(24,6))         AS naive_total
            FROM {schema}.orders o
            LEFT JOIN lines l ON l.l_orderkey = o.o_orderkey
        )
        SELECT
            *,
            recorded_total - billed_total AS diff,
            recorded_total - naive_total  AS naive_diff,
            CASE
                WHEN line_count = 0 THEN 'no_line_items'
                WHEN ABS(recorded_total - billed_total) > {ORDER_TOTAL_TOLERANCE} THEN 'total_mismatch'
            END AS mismatch_reason
        FROM joined
    """)


def order_stats(recon):
    return recon.agg(
        F.count("*").alias("orders"),
        F.sum(F.when(F.col("mismatch_reason").isNotNull(), 1).otherwise(0)).alias("mismatched"),
        F.sum(F.when(F.col("line_count") == 0, 1).otherwise(0)).alias("no_lines"),
        F.max(F.abs("diff")).alias("max_abs_diff"),
        F.sum(F.when(F.col("mismatch_reason").isNotNull(), F.abs("diff"))).alias("mismatched_amount"),
        F.sum(F.when(F.abs("naive_diff") > ORDER_TOTAL_TOLERANCE, 1).otherwise(0)).alias("naive_mismatched"),
        F.max(F.abs("naive_diff")).alias("naive_max_abs_diff"),
    ).first()

In [ ]:
LATEST_TABLE = f"{MON}.mismatched_orders_latest"
LOG_TABLE    = f"{MON}.mismatched_orders_log"
METRIC_TABLE = f"{MON}.metric_mismatched_orders"

if "order_total" in GROUPS:
    G = "order_total"
    RULE_MATCH    = "Order total = sum of its line items (billed in cents, ±$0.01)"
    RULE_NO_LINES = "Every order has at least one line item"
    EXPECTED      = f"|o_totalprice − Σ billed line totals| ≤ ${ORDER_TOTAL_TOLERANCE}"

    # --- bronze: raw data, warning only
    if table_exists(f"{BRONZE}.orders") and table_exists(f"{BRONZE}.lineitem"):
        b = order_stats(order_reconciliation(BRONZE))
        record("V1.1", G, "bronze", RULE_MATCH, "warn", b["orders"], b["mismatched"],
               f"max |diff| ${money(b['max_abs_diff'])}", EXPECTED)
        record("V1.2", G, "bronze", RULE_NO_LINES, "warn", b["orders"], b["no_lines"])
    else:
        record_missing("V1.1", G, "bronze", RULE_MATCH, f"{BRONZE}.orders / lineitem")

    # --- silver: the layer the business reads
    if table_exists(f"{SILVER}.orders") and table_exists(f"{SILVER}.lineitem"):
        recon = order_reconciliation(SILVER)

        # Persist the current mismatches first (one scan), then aggregate (second scan).
        # Serverless compute has no DataFrame cache, so we avoid recomputing the join more than twice.
        (recon.filter("mismatch_reason IS NOT NULL")
            .select(
                F.lit(RUN_ID).alias("run_id"), F.lit(RUN_TS).alias("run_ts"), F.lit(RUN_LABEL).alias("run_label"),
                "order_key", "order_date", "recorded_total", "billed_total", "diff", "line_count", "mismatch_reason",
            )
            .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(LATEST_TABLE))

        s = order_stats(recon)
        record("V1.1", G, "silver", RULE_MATCH, "error", s["orders"], s["mismatched"],
               f"max |diff| ${money(s['max_abs_diff'])}", EXPECTED)
        record("V1.2", G, "silver", RULE_NO_LINES, "error", s["orders"], s["no_lines"])
        naive_pct = 100.0 * (s["naive_mismatched"] or 0) / max(s["orders"], 1)
        record("V1.3", G, "silver", "Same check with the naive formula (no cent rounding) — shows why billing rounding matters",
               "info", s["orders"], s["naive_mismatched"],
               f"{naive_pct:.1f}% of orders off by > $0.01, max |diff| ${money(s['naive_max_abs_diff'])}",
               "informational")

        # --- new vs. previous run (the alert condition)
        prev_run = None
        if table_exists(METRIC_TABLE):
            prev = spark.table(METRIC_TABLE).orderBy(F.desc("run_ts")).select("run_id").first()
            prev_run = prev["run_id"] if prev else None

        current_keys = spark.table(LATEST_TABLE).select("order_key")
        if prev_run is not None and table_exists(LOG_TABLE):
            prev_keys = spark.table(LOG_TABLE).filter(F.col("run_id") == prev_run).select("order_key")
        else:
            prev_keys = spark.createDataFrame([], "order_key BIGINT")

        new_keys      = current_keys.join(prev_keys, "order_key", "left_anti")
        new_count     = new_keys.count()
        resolved      = prev_keys.join(current_keys, "order_key", "left_anti").count()
        new_sample    = [r["order_key"] for r in new_keys.orderBy("order_key").limit(10).collect()]

        # Only now add this run to the log, so the comparison above used the previous run only.
        spark.table(LATEST_TABLE).write.mode("append").saveAsTable(LOG_TABLE)

        METRIC = {
            "run_id": RUN_ID, "run_ts": RUN_TS, "run_label": RUN_LABEL,
            "orders_checked": int(s["orders"]),
            "mismatched_orders": int(s["mismatched"] or 0),
            "orders_without_lines": int(s["no_lines"] or 0),
            "mismatched_amount": Decimal(s["mismatched_amount"] or 0).quantize(Decimal("0.01")),
            "new_mismatched_orders": int(new_count),
            "resolved_mismatched_orders": int(resolved),
            "naive_mismatched_orders": int(s["naive_mismatched"] or 0),
            "alert": new_count > 0,
            "new_order_keys_sample": new_sample,
            "previous_run_id": prev_run,
        }
        spark.createDataFrame([METRIC], """
            run_id STRING, run_ts TIMESTAMP, run_label STRING,
            orders_checked BIGINT, mismatched_orders BIGINT, orders_without_lines BIGINT,
            mismatched_amount DECIMAL(38,2), new_mismatched_orders BIGINT, resolved_mismatched_orders BIGINT,
            naive_mismatched_orders BIGINT, alert BOOLEAN, new_order_keys_sample ARRAY<BIGINT>, previous_run_id STRING
        """).write.mode("append").saveAsTable(METRIC_TABLE)

        print()
        print(f"Mismatched orders now: {METRIC['mismatched_orders']:,}   new since previous run: {new_count:,}   resolved: {resolved:,}")
    else:
        record_missing("V1.1", G, "silver", RULE_MATCH, f"{SILVER}.orders / lineitem")
else:
    print("Skipped: order_total")

In [ ]:
# Every order that fails V1 right now (empty when everything reconciles).
if "order_total" in GROUPS and table_exists(LATEST_TABLE):
    display(spark.table(LATEST_TABLE).orderBy(F.desc(F.abs("diff"))).limit(100))

## V2 — Discount and tax inside their allowed range

Bounds: `l_discount ∈ [0.00, 0.10]`, `l_tax ∈ [0.00, 0.08]`. They were derived by profiling the source (min/max and the full distinct-value histogram: every 0.01 step populated with a near-uniform count) and match the TPC-H specification, clause 4.2.3. The justification is in the README.

The rule is checked on bronze (warning: raw data), silver and gold (errors). `NULL` counts as out of range. The observed min, max and number of distinct values are stored with each result, so a change in the source domain shows up in the history even while the rule still passes.

V2.3 checks that the bounds declared as `CHECK` constraints on `silver.lineitem` are the same as here.

In [ ]:
if "ranges" in GROUPS:
    G = "ranges"
    RULE_DISC = f"l_discount within [{DISCOUNT_MIN}, {DISCOUNT_MAX}]"
    RULE_TAX  = f"l_tax within [{TAX_MIN}, {TAX_MAX}]"

    RANGE_TARGETS = [
        # (layer, table, discount column, tax column)
        ("bronze", f"{BRONZE}.lineitem",    "l_discount", "l_tax"),
        ("silver", f"{SILVER}.lineitem",    "l_discount", "l_tax"),
        ("gold",   f"{GOLD}.fact_lineitem", "discount",   "tax"),
    ]

    def out_of_range(col, lo, hi):
        # between(...) is NULL for a NULL value -> otherwise(1): NULL counts as a violation
        return F.sum(F.when(F.col(col).between(lo, hi), 0).otherwise(1))

    for layer, table, dcol, tcol in RANGE_TARGETS:
        if not table_exists(table):
            record_missing("V2.1", G, layer, RULE_DISC, table)
            continue
        r = spark.table(table).agg(
            F.count("*").alias("n"),
            out_of_range(dcol, DISCOUNT_MIN, DISCOUNT_MAX).alias("d_bad"),
            F.min(dcol).alias("d_min"), F.max(dcol).alias("d_max"), F.countDistinct(dcol).alias("d_distinct"),
            out_of_range(tcol, TAX_MIN, TAX_MAX).alias("t_bad"),
            F.min(tcol).alias("t_min"), F.max(tcol).alias("t_max"), F.countDistinct(tcol).alias("t_distinct"),
        ).first()
        sev = LAYER_SEVERITY[layer]
        record("V2.1", G, layer, RULE_DISC, sev, r["n"], r["d_bad"],
               f"observed [{r['d_min']}, {r['d_max']}], {r['d_distinct']} distinct values",
               f"[{DISCOUNT_MIN}, {DISCOUNT_MAX}]")
        record("V2.2", G, layer, RULE_TAX, sev, r["n"], r["t_bad"],
               f"observed [{r['t_min']}, {r['t_max']}], {r['t_distinct']} distinct values",
               f"[{TAX_MIN}, {TAX_MAX}]")

    # V2.3 — the CHECK constraints registered on silver use the same bounds
    RULE_DECL = "Silver CHECK constraints declare the same discount/tax bounds"
    if table_exists(f"{SILVER}.lineitem"):
        props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {SILVER}.lineitem").collect()}
        expected_constraints = {
            "l_discount_range": [DISCOUNT_MIN, DISCOUNT_MAX],
            "l_tax_range":      [TAX_MIN, TAX_MAX],
        }
        problems = []
        for name, bounds in expected_constraints.items():
            expr = props.get(f"delta.constraints.{name}")
            if expr is None:
                problems.append(f"{name}: not declared")
            elif sorted(Decimal(x) for x in re.findall(r"-?\d+\.\d+", expr)) != bounds:
                problems.append(f"{name}: '{expr}'")
        record("V2.3", G, "silver", RULE_DECL, "warn", len(expected_constraints), len(problems),
               "; ".join(problems) or "both constraints match", "same bounds as this notebook")
    else:
        record_missing("V2.3", G, "silver", RULE_DECL, f"{SILVER}.lineitem")
else:
    print("Skipped: ranges")

## V3 — Quantity, extended price and retail price are strictly positive

`NULL` is not positive, so it counts as a violation. Checked on bronze (warning), silver and gold (errors). In gold the columns are renamed (`quantity`, `extended_price`, `dim_part.retail_price`); the rule ID stays the same so the layers line up in the results.

In [ ]:
if "positive" in GROUPS:
    G = "positive"
    POSITIVE_RULES = {
        # column -> (rule id, readable name)
        "l_quantity":      ("V3.1", "Quantity > 0"),
        "quantity":        ("V3.1", "Quantity > 0"),
        "l_extendedprice": ("V3.2", "Extended price > 0"),
        "extended_price":  ("V3.2", "Extended price > 0"),
        "p_retailprice":   ("V3.3", "Retail price > 0"),
        "retail_price":    ("V3.3", "Retail price > 0"),
    }
    POSITIVE_TARGETS = [
        ("bronze", f"{BRONZE}.lineitem",    ["l_quantity", "l_extendedprice"]),
        ("bronze", f"{BRONZE}.part",        ["p_retailprice"]),
        ("silver", f"{SILVER}.lineitem",    ["l_quantity", "l_extendedprice"]),
        ("silver", f"{SILVER}.part",        ["p_retailprice"]),
        ("gold",   f"{GOLD}.fact_lineitem", ["quantity", "extended_price"]),
        ("gold",   f"{GOLD}.dim_part",      ["retail_price"]),
    ]

    for layer, table, cols in POSITIVE_TARGETS:
        if not table_exists(table):
            for c in cols:
                rid, name = POSITIVE_RULES[c]
                record_missing(rid, G, layer, name, table)
            continue
        aggs = [F.count("*").alias("n")]
        for c in cols:
            aggs += [F.sum(F.when(F.col(c) > 0, 0).otherwise(1)).alias(f"{c}__bad"), F.min(c).alias(f"{c}__min")]
        r = spark.table(table).agg(*aggs).first()
        for c in cols:
            rid, name = POSITIVE_RULES[c]
            record(rid, G, layer, name, LAYER_SEVERITY[layer], r["n"], r[f"{c}__bad"],
                   f"min {c} = {r[f'{c}__min']}", "> 0")
else:
    print("Skipped: positive")

## V4 — Money reconciles across layers

The **same** figures are computed in every layer and compared:

| Measure | Definition |
|---|---|
| `lineitem_rows` | number of line items |
| `gross_revenue` | `Σ l_extendedprice` |
| `net_revenue` | `Σ l_extendedprice × (1 − l_discount)`, the project's revenue definition. In gold this is the stored `fact_lineitem.net_revenue` column, not a recomputation. |
| `order_rows`, `order_totalprice` | number of orders and `Σ o_totalprice` (gold has no order-total column, by design: repeating it on every line would double-count) |

Expected equalities, each within **$0.01** (row counts exactly):

| Rule | Equation | What a failure means |
|---|---|---|
| V4.1 | source = bronze | bronze is not a faithful copy |
| V4.2 | bronze = silver + quarantine (+ exact duplicates dropped) | silver lost or invented rows/money without quarantining them |
| V4.3 | silver = gold | the gold build dropped, duplicated or changed line items |
| V4.4 | silver = gold **per order year** | totals agree only by accident (e.g. dates shifted between years) |
| V4.5 | in every gold row: `net = gross − discount_amount` and `net_with_tax = net + tax_amount` | the three revenue figures of Q2 are not consistent with each other |

The quarantine side reads the money columns from the original bronze row stored as JSON in `silver.quarantine.record`.

In [ ]:
RECON_ROWS = []

if "reconciliation" in GROUPS:
    G = "reconciliation"
    MEASURES = ["lineitem_rows", "gross_revenue", "net_revenue", "order_rows", "order_totalprice"]
    NET_SQL  = "SUM(CAST(l_extendedprice * (1 - l_discount) AS DECIMAL(38,4)))"   # same formula as gold

    def raw_totals(schema, distinct=False):
        """Measures for a layer with TPC-H column names (source, bronze, silver)."""
        if not (table_exists(f"{schema}.lineitem") and table_exists(f"{schema}.orders")):
            return None
        li = spark.table(f"{schema}.lineitem")
        od = spark.table(f"{schema}.orders")
        if distinct:  # silver drops exact duplicate rows before applying its rules
            li, od = li.dropDuplicates(), od.dropDuplicates()
        l = li.agg(
            F.count("*").alias("rows"),
            F.sum(F.col("l_extendedprice").cast("decimal(38,2)")).alias("gross"),
            F.expr(NET_SQL).alias("net"),
        ).first()
        o = od.agg(F.count("*").alias("rows"), F.sum(F.col("o_totalprice").cast("decimal(38,2)")).alias("total")).first()
        return {"lineitem_rows": l["rows"], "gross_revenue": l["gross"], "net_revenue": l["net"],
                "order_rows": o["rows"], "order_totalprice": o["total"]}

    def quarantine_totals():
        q_table = f"{SILVER}.quarantine"
        zero = {m: 0 for m in MEASURES}
        if not table_exists(q_table):
            return zero
        q = spark.table(q_table)
        l = (q.filter("source_table = 'lineitem'")
               .select(F.expr("try_cast(get_json_object(record, '$.l_extendedprice') AS DECIMAL(18,2))").alias("l_extendedprice"),
                       F.expr("try_cast(get_json_object(record, '$.l_discount') AS DECIMAL(18,2))").alias("l_discount"))
               .agg(F.count("*").alias("rows"),
                    F.sum(F.col("l_extendedprice").cast("decimal(38,2)")).alias("gross"),
                    F.expr(NET_SQL).alias("net"))
               .first())
        o = (q.filter("source_table = 'orders'")
               .select(F.expr("try_cast(get_json_object(record, '$.o_totalprice') AS DECIMAL(18,2))").alias("t"))
               .agg(F.count("*").alias("rows"), F.sum(F.col("t").cast("decimal(38,2)")).alias("total"))
               .first())
        return {"lineitem_rows": l["rows"], "gross_revenue": l["gross"] or 0, "net_revenue": l["net"] or 0,
                "order_rows": o["rows"], "order_totalprice": o["total"] or 0}

    def gold_totals():
        if not table_exists(f"{GOLD}.fact_lineitem"):
            return None
        g = spark.table(f"{GOLD}.fact_lineitem").agg(
            F.count("*").alias("rows"), F.sum("gross_revenue").alias("gross"), F.sum("net_revenue").alias("net"),
        ).first()
        return {"lineitem_rows": g["rows"], "gross_revenue": g["gross"], "net_revenue": g["net"],
                "order_rows": None, "order_totalprice": None}

    def duplicates_dropped():
        """Exact duplicates silver dropped in its latest load, from silver.load_log (None = unknown)."""
        log = f"{SILVER}.load_log"
        if not table_exists(log):
            return None
        rows = (spark.table(log).groupBy("table_name")
                .agg(F.max_by("duplicate_rows_dropped", "run_ts").alias("d")).collect())
        return {r["table_name"]: r["d"] for r in rows}

    source = raw_totals(SOURCE) if table_exists(f"{SOURCE}.lineitem") else None
    bronze = raw_totals(BRONZE)
    silver = raw_totals(SILVER)
    gold   = gold_totals()
    quar   = quarantine_totals()

    # bronze as silver saw it: without exact duplicates (only recomputed if silver reported any)
    dupes = duplicates_dropped()
    if bronze is None:
        bronze_dedup = None
    elif dupes is not None and dupes.get("lineitem", 0) == 0 and dupes.get("orders", 0) == 0:
        bronze_dedup = bronze
    else:
        bronze_dedup = raw_totals(BRONZE, distinct=True)

    def d(x):
        return None if x is None else Decimal(x)

    def diff(a, b):
        return None if a is None or b is None else d(a) - d(b)

    def mismatched(measures, pairs):
        """Count measures whose two sides differ (rows exactly, money beyond MONEY_TOLERANCE)."""
        bad, notes = 0, []
        for m in measures:
            a, b = pairs(m)
            delta = diff(a, b)
            limit = Decimal(0) if m.endswith("_rows") else MONEY_TOLERANCE
            if delta is None or abs(delta) > limit:
                bad += 1
                notes.append(f"{m}: {money(a) if not m.endswith('_rows') else a} vs {money(b) if not m.endswith('_rows') else b}")
        return bad, notes

    # V4.1 source = bronze
    if source is None:
        record("V4.1", G, "bronze", "Source = bronze (row counts and money)", "info", 0, 0,
               f"source {SOURCE} not reachable from this workspace — skipped")
    elif bronze is None:
        record_missing("V4.1", G, "bronze", "Source = bronze (row counts and money)", f"{BRONZE}.lineitem / orders")
    else:
        bad, notes = mismatched(MEASURES, lambda m: (source[m], bronze[m]))
        record("V4.1", G, "bronze", "Source = bronze (row counts and money)", "error", len(MEASURES), bad,
               "; ".join(notes) or f"net revenue {money(bronze['net_revenue'])} in both", "identical")

    # V4.2 bronze (deduplicated) = silver + quarantine
    if bronze_dedup is None or silver is None:
        record_missing("V4.2", G, "silver", "Bronze = silver + quarantine", f"{BRONZE} or {SILVER}")
    else:
        bad, notes = mismatched(MEASURES, lambda m: (bronze_dedup[m], d(silver[m]) + d(quar[m])))
        record("V4.2", G, "silver", "Bronze = silver + quarantine (+ exact duplicates dropped)", "error", len(MEASURES), bad,
               "; ".join(notes) or f"quarantined: {quar['lineitem_rows']:,} line items ({money(quar['net_revenue'])} net), "
                                   f"{quar['order_rows']:,} orders",
               f"within ${MONEY_TOLERANCE}")

    # V4.3 silver = gold
    LINE_MEASURES = ["lineitem_rows", "gross_revenue", "net_revenue"]
    if silver is None or gold is None:
        record_missing("V4.3", G, "gold", "Silver = gold (line items, gross and net revenue)", f"{SILVER} or {GOLD}.fact_lineitem")
    else:
        bad, notes = mismatched(LINE_MEASURES, lambda m: (silver[m], gold[m]))
        record("V4.3", G, "gold", "Silver = gold (line items, gross and net revenue)", "error", len(LINE_MEASURES), bad,
               "; ".join(notes) or f"net revenue {money(gold['net_revenue'])} in both", f"within ${MONEY_TOLERANCE}")

    # V4.4 silver = gold, per order year
    if silver is not None and gold is not None:
        by_year = spark.sql(f"""
            WITH s AS (
                SELECT YEAR(o.o_orderdate) AS order_year,
                       SUM(CAST(l.l_extendedprice * (1 - l.l_discount) AS DECIMAL(38,4))) AS silver_net
                FROM {SILVER}.lineitem l
                JOIN {SILVER}.orders o ON o.o_orderkey = l.l_orderkey
                GROUP BY YEAR(o.o_orderdate)
            ),
            g AS (
                SELECT YEAR(order_date) AS order_year, SUM(net_revenue) AS gold_net
                FROM {GOLD}.fact_lineitem
                GROUP BY YEAR(order_date)
            )
            SELECT COALESCE(s.order_year, g.order_year) AS order_year, s.silver_net, g.gold_net,
                   g.gold_net - s.silver_net AS diff
            FROM s FULL OUTER JOIN g ON s.order_year = g.order_year
            ORDER BY order_year
        """).collect()
        bad_years = [r["order_year"] for r in by_year
                     if r["diff"] is None or abs(r["diff"]) > MONEY_TOLERANCE]
        record("V4.4", G, "gold", "Silver = gold net revenue for every order year", "error", len(by_year), len(bad_years),
               f"years off: {bad_years}" if bad_years else f"{len(by_year)} years match", f"within ${MONEY_TOLERANCE} per year")

    # V4.5 gold revenue measures are consistent with each other, row by row
    if gold is not None:
        r = spark.table(f"{GOLD}.fact_lineitem").agg(
            F.count("*").alias("n"),
            F.sum(F.when(
                (F.abs(F.col("net_revenue") - (F.col("gross_revenue") - F.col("discount_amount"))) > Decimal("0.0001"))
                | (F.abs(F.col("net_revenue_with_tax") - (F.col("net_revenue") + F.col("tax_amount"))) > Decimal("0.0001")),
                1).otherwise(0)).alias("bad"),
        ).first()
        record("V4.5", G, "gold", "Gold: net = gross − discount and net_with_tax = net + tax, per row", "error",
               r["n"], r["bad"], "", "exact to 4 decimals")

    # Store the layer-by-layer figures for the monitoring dashboard
    for m in MEASURES:
        def val(layer_totals):
            return None if layer_totals is None or layer_totals[m] is None else Decimal(layer_totals[m])
        dup_value = None
        if bronze is not None and bronze_dedup is not None:
            dup_value = diff(bronze[m], bronze_dedup[m])
        RECON_ROWS.append({
            "run_id": RUN_ID, "run_ts": RUN_TS, "run_label": RUN_LABEL, "measure": m,
            "source_value": val(source), "bronze_value": val(bronze), "duplicates_dropped_value": dup_value,
            "quarantine_value": Decimal(quar[m] or 0), "silver_value": val(silver), "gold_value": val(gold),
        })
else:
    print("Skipped: reconciliation")

In [ ]:
RECON_SCHEMA = """
    run_id STRING, run_ts TIMESTAMP, run_label STRING, measure STRING,
    source_value DECIMAL(38,4), bronze_value DECIMAL(38,4), duplicates_dropped_value DECIMAL(38,4),
    quarantine_value DECIMAL(38,4), silver_value DECIMAL(38,4), gold_value DECIMAL(38,4)
"""
if RECON_ROWS:
    recon_df = (spark.createDataFrame(RECON_ROWS, RECON_SCHEMA)
        .withColumn("bronze_minus_source", F.col("bronze_value") - F.col("source_value"))
        .withColumn("bronze_minus_silver_quarantine_dupes",
                    F.col("bronze_value") - F.col("silver_value") - F.col("quarantine_value")
                    - F.coalesce(F.col("duplicates_dropped_value"), F.lit(0)))
        .withColumn("silver_minus_gold", F.col("silver_value") - F.col("gold_value")))
    recon_df.write.mode("append").option("mergeSchema", "true").saveAsTable(f"{MON}.layer_reconciliation")
    measure_rank = F.array_position(F.array(*[F.lit(m) for m in MEASURES]), F.col("measure"))
    display(recon_df.orderBy(measure_rank).drop("run_id", "run_ts", "run_label"))

## Results

All rule results of this run, stored in `monitoring.dq_results`. **FAIL** = an error rule on silver/gold failed; **WARN** = a problem in the raw layers (or a declaration drift) that does not stop the pipeline; **INFO** = a reported number without pass/fail.

In [ ]:
RESULT_SCHEMA = """
    run_id STRING, run_ts TIMESTAMP, run_label STRING, rule_id STRING, rule_group STRING, layer STRING,
    rule STRING, severity STRING, status STRING, checked_rows BIGINT, failed_rows BIGINT,
    observed STRING, expected STRING
"""
results_df = spark.createDataFrame(RESULTS, RESULT_SCHEMA)
results_df.write.mode("append").saveAsTable(f"{MON}.dq_results")

layer_rank = F.create_map(*[x for k, v in LAYER_ORDER.items() for x in (F.lit(k), F.lit(v))])
display(
    results_df
    .withColumn("_layer_rank", layer_rank[F.col("layer")])
    .orderBy("rule_id", "_layer_rank")
    .select("rule_id", "layer", "status", "rule", "failed_rows", "checked_rows", "observed", "expected")
)

counts = {s: sum(1 for r in RESULTS if r["status"] == s) for s in ("PASS", "WARN", "FAIL", "INFO")}
print(f"PASS {counts['PASS']}   WARN {counts['WARN']}   FAIL {counts['FAIL']}   INFO {counts['INFO']}")

## Verdict and alert

- **Alert:** a mismatched order appeared that was not mismatched in the previous run (`metric_mismatched_orders.alert`). The same flag drives the Databricks SQL alert described in `05_monitoring`.
- **Gate:** if any *error* rule failed and `fail_on_error = true`, this cell raises. In a Job that fails the run, which stops downstream tasks and sends the Job's failure notification.
- Otherwise the notebook exits with a JSON summary (used by `06_alert_demo`).

In [ ]:
failed_rules = [f"{r['rule_id']} ({r['layer']}): {r['rule']}" for r in RESULTS if r["status"] == "FAIL"]
summary = {
    "run_id": RUN_ID, "run_ts": str(RUN_TS), "run_label": RUN_LABEL, "groups": GROUPS,
    **{k.lower(): v for k, v in counts.items()},
    "failed_rules": failed_rules,
}
if METRIC is not None:
    summary.update({k: METRIC[k] for k in
                    ("mismatched_orders", "new_mismatched_orders", "resolved_mismatched_orders", "alert", "new_order_keys_sample")})

if METRIC is not None and METRIC["alert"]:
    print(f"🚨 ALERT: {METRIC['new_mismatched_orders']:,} new mismatched order(s) since the previous run, "
          f"e.g. order_key {METRIC['new_order_keys_sample'][:5]}. See {LATEST_TABLE}.")
else:
    print("No new mismatched orders since the previous run.")

if failed_rules:
    print("\nFailed error rules:\n  - " + "\n  - ".join(failed_rules))
    if FAIL_ON_ERROR:
        raise RuntimeError(f"Finance validation failed ({len(failed_rules)} error rule(s)); see {MON}.dq_results, run_id={RUN_ID}")
else:
    print("\nAll error rules passed: the Finance numbers in silver and gold can be trusted.")

dbutils.notebook.exit(json.dumps(summary, default=str))